## Ejercicio 1 - Pandas

A) Dar el output de ``` df.groupby("curso")["nota].mean()``` y de ``` df[df["nota"]>=80].shape[0]```



B) Escribir la linea de pandas que responda ¿Cuál es la nota máxima por curso

### A) Construcción del DataFrame

In [17]:
import pandas as pd

df = pd.DataFrame({
    "curso": ["mate", "mate", "prog", "prog", "prog", "mate"],
    "nota":  [80, 60, 90, 70, 95, 70],
})

df

,curso,nota
0,mate,80
1,mate,60
2,prog,90
3,prog,70
4,prog,95
5,mate,70


#### A.1) `df.groupby("curso")["nota"].mean()`
- `mate`: (80 + 60 + 70) / 3 = **70.0**
- `prog`: (90 + 70 + 95) / 3 = **85.0**

In [18]:
df.groupby("curso")["nota"].mean()

curso
mate    70.0
prog    85.0
Name: nota, dtype: float64

#### A.2) `df[df["nota"] >= 80].shape[0]`

- Notas que cumplen: 80, 90, 95 → **3**

In [19]:
# La máscara booleana, paso intermedio para ver qué está pasando
df["nota"] >= 80

0     True
1    False
2     True
3    False
4     True
5    False
Name: nota, dtype: bool

In [20]:
df[df["nota"] >= 80].shape[0]

3

## Ejercicio 2 - Naive Bayes con Laplace

Corpus con priors iguales. Conteos: "gratis" aparece 3 veces en spam y 0 en normal; "hola", 0 en spam y 8 en normal. Total de palabras: 8 en spam y 8 en normal. Vocabulario: $|V| = 5$. Con $\alpha = 1$, recordar:

$$P(\text{palabra} \mid \text{clase}) = \frac{\text{conteo} + \alpha}{\text{total} + \alpha |V|}$$

A) Calcular las cuatro probabilidades suavizadas y clasificar el mensaje "gratis hola", mostrando las dos cantidades que compara.

B) En una frase: ¿qué habría pasado sin Laplace?

### A) Probabilidades suavizadas

Primero declaro los datos del enunciado como variables y escribo la fórmula de Laplace como función.

In [21]:
from fractions import Fraction

# Datos del enunciado
alpha = 1
V = 5                      # tamaño del vocabulario

total = {"spam": 8, "normal": 8}

conteo = {
    "gratis": {"spam": 3, "normal": 0},
    "hola":   {"spam": 0, "normal": 8},
}


def p_laplace(palabra, clase):
    """P(palabra | clase) = (conteo + alpha) / (total + alpha * |V|)"""
    numerador = conteo[palabra][clase] + alpha
    denominador = total[clase] + alpha * V
    return Fraction(numerador, denominador)

In [22]:
p_gratis_spam = p_laplace("gratis", "spam")

print(f"P(gratis | spam) = {p_gratis_spam} = {float(p_gratis_spam):.4f}")

P(gratis | spam) = 4/13 = 0.3077


**Las otras tres probabilidades**, con el mismo denominador $8 + 1 \cdot 5 = 13$:

- $P(\text{gratis} \mid \text{normal}) = \frac{0 + 1}{13} = \frac{1}{13}$
- $P(\text{hola} \mid \text{spam}) = \frac{0 + 1}{13} = \frac{1}{13}$
- $P(\text{hola} \mid \text{normal}) = \frac{8 + 1}{13} = \frac{9}{13}$

Las dos con conteo 0 dan lo mismo: Laplace las levanta de $0$ a $\frac{1}{13}$.

In [23]:
p_gratis_normal = p_laplace("gratis", "normal")
p_hola_spam     = p_laplace("hola", "spam")
p_hola_normal   = p_laplace("hola", "normal")

for nombre, p in [
    ("P(gratis | spam)  ", p_gratis_spam),
    ("P(gratis | normal)", p_gratis_normal),
    ("P(hola   | spam)  ", p_hola_spam),
    ("P(hola   | normal)", p_hola_normal),
]:
    print(f"{nombre} = {str(p):>5} = {float(p):.4f}")

P(gratis | spam)   =  4/13 = 0.3077
P(gratis | normal) =  1/13 = 0.0769
P(hola   | spam)   =  1/13 = 0.0769
P(hola   | normal) =  9/13 = 0.6923


### Cantidades a comparar para "gratis hola"

In [24]:
prior = {"spam": Fraction(1, 2), "normal": Fraction(1, 2)}

score_spam   = prior["spam"]   * p_gratis_spam   * p_hola_spam
score_normal = prior["normal"] * p_gratis_normal * p_hola_normal

print(f"score(spam)   = 1/2 * {p_gratis_spam} * {p_hola_spam} = {score_spam} = {float(score_spam):.5f}")
print(f"score(normal) = 1/2 * {p_gratis_normal} * {p_hola_normal} = {score_normal} = {float(score_normal):.5f}")

score(spam)   = 1/2 * 4/13 * 1/13 = 2/169 = 0.01183
score(normal) = 1/2 * 1/13 * 9/13 = 9/338 = 0.02663


A. Comparando lso dos scores obtenidos, vemos que el score para que "gratis hola" sea categorizada como spam es de 0.01183, o un ≈1.18%, mientras que su score para que caiga en la categoria normal es de 0.02663, o un ≈2.66%. Por lo tanto, se decide como el numero mas grande, asi que podriamos categorizar el mensaje "gratis hola" como normal. 

B. De no haber aplicado el suavizado de LaPlace, el numerador de ``` P(hola | spam) y a P(gratis | normal) ``` con ⍺=0 se hubiera anulado y habriamos anulado por completo la clase, como cada score es un producto, un solo cero anula todo el producto de esa clase, sin importar la evidencia de las otras palabras

## Ejercicio 3 - k-NN a mano

Punto nuevo: $(2, 3)$. Datos: $P_1(1, 3)$ clase A; $P_2(2, 5)$ clase B; $P_3(4, 3)$ clase B; $P_4(1, 1)$ clase A.

A) Calcular las cuatro distancias euclidianas (pueden quedar como raíces).

B) Clasificar con $k = 3$ y con $k = 1$, mostrando el voto.

C) En una frase: si la primera coordenada estuviera en quetzales (miles) y la segunda en años, ¿qué pasaría con los vecinos y qué haría antes de correr k-NN?

### A) Distancias euclidianas

Distancia del punto nuevo $q = (2, 3)$ a cada $P_i = (x_i, y_i)$:

$$d(q, P_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}$$


In [25]:
from math import sqrt

# Punto nuevo
xq, yq = 2, 3


def mostrar(nombre, x, y, clase, dx2, dy2, d):
    """Solo formatea: la aritmética se hace afuera, paso a paso."""
    suma = dx2 + dy2
    signo = "=" if d == int(d) else "≈"
    print(f"{nombre} ({x}, {y})  clase {clase}   "
          f"√(({xq}−{x})² + ({yq}−{y})²) = √({dx2} + {dy2}) = √{suma:<2} {signo} {d:.4f}")


print(f"Punto nuevo q = ({xq}, {yq})\n")

# P1 = (1, 3), clase A
dx2, dy2 = (xq - 1)**2, (yq - 3)**2
d_p1 = sqrt(dx2 + dy2)
mostrar("P1", 1, 3, "A", dx2, dy2, d_p1)

# P2 = (2, 5), clase B
dx2, dy2 = (xq - 2)**2, (yq - 5)**2
d_p2 = sqrt(dx2 + dy2)
mostrar("P2", 2, 5, "B", dx2, dy2, d_p2)

# P3 = (4, 3), clase B
dx2, dy2 = (xq - 4)**2, (yq - 3)**2
d_p3 = sqrt(dx2 + dy2)
mostrar("P3", 4, 3, "B", dx2, dy2, d_p3)

# P4 = (1, 1), clase A
dx2, dy2 = (xq - 1)**2, (yq - 1)**2
d_p4 = sqrt(dx2 + dy2)
mostrar("P4", 1, 1, "A", dx2, dy2, d_p4)

Punto nuevo q = (2, 3)

P1 (1, 3)  clase A   √((2−1)² + (3−3)²) = √(1 + 0) = √1  = 1.0000
P2 (2, 5)  clase B   √((2−2)² + (3−5)²) = √(0 + 4) = √4  = 2.0000
P3 (4, 3)  clase B   √((2−4)² + (3−3)²) = √(4 + 0) = √4  = 2.0000
P4 (1, 1)  clase A   √((2−1)² + (3−1)²) = √(1 + 4) = √5  ≈ 2.2361


### B) Clasificación con $k = 3$ y $k = 1$

k-NN ordena los puntos por distancia, toma los $k$ más cercanos y cada uno vota por su clase;
gana la mayoría. Uso las distancias `d_p1 … d_p4` calculadas en A.

In [26]:
from collections import Counter

# (distancia, punto, clase) usando las distancias de la parte A
vecinos = [
    (d_p1, "P1", "A"),
    (d_p2, "P2", "B"),
    (d_p3, "P3", "B"),
    (d_p4, "P4", "A"),
]

# Ordenar de más cercano a más lejano
vecinos_ordenados = sorted(vecinos)

print("Orden por distancia:")
for d, nombre, clase in vecinos_ordenados:
    print(f"  {nombre}  d = {d:.4f}  clase {clase}")


def votar(k):
    k_cercanos = vecinos_ordenados[:k]
    votos = Counter(clase for _, _, clase in k_cercanos)
    ganadora = votos.most_common(1)[0][0]

    print(f"\nk = {k}")
    print("  vecinos:", ", ".join(f"{n} ({c})" for _, n, c in k_cercanos))
    print("  voto:   ", ", ".join(f"{c} = {v}" for c, v in sorted(votos.items())))
    print(f"  clase:   {ganadora}")
    return ganadora


clase_k3 = votar(3)
clase_k1 = votar(1)

Orden por distancia:
  P1  d = 1.0000  clase A
  P2  d = 2.0000  clase B
  P3  d = 2.0000  clase B
  P4  d = 2.2361  clase A

k = 3
  vecinos: P1 (A), P2 (B), P3 (B)
  voto:    A = 1, B = 2
  clase:   B

k = 1
  vecinos: P1 (A)
  voto:    A = 1
  clase:   A


### Ejercicio C. En una frase: si la primera coordenada estuviera en quetzales (miles) y la segunda en años, ¿qué pasaría con los vecinos y qué haría antes de correr k-NN?

Como las distancias euclidanas suman las diferencias al cuadrado, la diferencia de miles de quetzales dominaría sobrw las dierencias de los años, esto debido a las unidades en las que se trabajan ambas variables. Por esto mismo, es necesario escalar ambas variables antes de correr k-NN para estandarizar las unidades y darle peso a la variable realmente influyente. Si las operamos sin escalarlas, los vecinos se eligrían solo por la primera coordenada.

## Bitacora

### Ejercicio 1
- En el ejercicio 1, se utilizo el agente LLM Claude (Fable 5.1) para escribir el codigo que estaba descrito en la pregunta en python, para luego manualmente correrlos y sacar una conclusion con base al analisis
- Para el punto B, se utilizo para repasar brevemente la sintaxis de pandas y asi contestar correctamente la pregunta.


### Ejercicio 2
- Se utilizó el modelo LLM agéntico para transcribir la pregunta del ejercicio 2 de la hoja al jupyteer notebook
- En el punto A, se utilizo para escribir el codigo base de una probabilidad, el resto fueron hechas a mano en base al codigo generado por el LLM para el primer caso

### Ejercicio 3
- Se utilizó el modelo LLM agéntico para transcribir la pregunta del ejercicio 3 de la hoja al jupyteer notebook
- Se utilizó el agente para validar la fórmula de la distancia que tenía en mente y escribirla en Markdown
- Le pedí al agente que me modificara el código de las raices para mejorar el print y la legibilidad de la impresión de los valores.  Las funciones para calcular cada lor de cada raiz fue escrito a mano, copiado y moficiado. 
- Se utilizo al agente para escribiri el codigo del `df` en el punto B
